# 0 データ読み込み

In [1]:
# ローデータを読み込む

import pandas as pd
import pandas as pd
import numpy as np

df_fixtures = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_fixtures.csv')
df_managers = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App//Data/RawData/df_managers.csv')
df_manager_changes = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_manager_changes.csv')
df_ranks = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_ranks.csv')
df_market_values = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_market_values.csv')
df_xg = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_xg.csv')
df_agi_kagi = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_agi_kagi.csv')
stadium_df = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/stadium_df.csv')
df_player_stats = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_player_stats.csv')
df_team_stats = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_team_stats.csv')
df_goal_patterns = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_goal_patterns.csv')
df_team_styles = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_team_styles.csv')
df_season_stats = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_season_stats.csv')

In [2]:
# df_ranksをlong formatに変換
df_ranks_long = df_ranks.melt(id_vars=['Team', 'Year'], var_name='Section', value_name='Rank')
df_ranks_long['Section'] = pd.to_numeric(df_ranks_long['Section'], errors='coerce')
df_ranks_long = df_ranks_long.dropna(subset=['Section'])
df_ranks_long['Section'] = df_ranks_long['Section'].astype(int)
df_ranks = df_ranks_long

In [3]:
ML_dataset = df_fixtures.copy()

In [4]:
# 基本のデータとなるML_datasetを整える。

# 天気をシンプルに分類する関数
def simplify_weather(weather):
    # 欠損値（nan）の処理
    if pd.isna(weather):
        return 'Unknown'
    
    weather = str(weather)
    
    # サッカーにおいて影響が大きい順に条件分岐するのがポイントです
    if '屋内' in weather:
        return 'Indoor'       # 札幌ドームなど
    elif '雪' in weather or '雷' in weather:
        return 'Bad_Weather'  # 荒天（大雪、雷雨など）
    elif '雨' in weather:
        return 'Rain'         # 「雨のち晴」などもピッチが濡れていると仮定してRainに含める
    elif '晴' in weather:
        return 'Sunny'        # 晴れ
    elif '曇' in weather or '霧' in weather:
        return 'Cloudy'       # 曇り・霧
    else:
        return 'Other'

# 新しい特徴量として適用（カラム名は実際のデータに合わせて適宜変更してください）
# Weather列が存在する場合のみ処理
ML_dataset['Weather'] = ML_dataset['Weather'].apply(simplify_weather)
# 変換後のカテゴリの分布を確認
print(ML_dataset['Weather'].value_counts())


Weather
Sunny          952
Cloudy         366
Rain           320
Indoor          86
Unknown         19
Bad_Weather      9
Name: count, dtype: int64


In [5]:
# 追加：Competition列は不要なので削除
ML_dataset.drop(columns=['Competition'], inplace=True)

# 追加：Sectionは「第1節第1日」→「1」への変換
ML_dataset['Section'] = ML_dataset['Section'].astype(str).str.extract(r'第([0-9０-９]+)節')[0]
fullwidth_nums = str.maketrans('０１２３４５６７８９', '0123456789')
ML_dataset['Section'] = ML_dataset['Section'].str.translate(fullwidth_nums)
ML_dataset['Section'] = pd.to_numeric(ML_dataset['Section'], errors='coerce').astype('Int64')

# 追加：Date_ParsedをDateに反映し、Date_Parsedを削除
ML_dataset['Date'] = pd.to_datetime(ML_dataset['Date_Parsed'], errors='coerce').dt.strftime('%Y-%m-%d')
ML_dataset.drop(columns=['Date_Parsed'], inplace=True)

print(ML_dataset[['Date', 'Section']].head())

         Date  Section
0  2021-02-26        1
1  2021-02-27        1
2  2021-02-27        1
3  2021-02-27        1
4  2021-02-27        1


In [6]:
# チーム名マッピング：日本語名→コード

# TEAMSディクショナリー（コード→日本語名）
TEAMS = {
    "kasm": "鹿島アントラーズ", "uraw": "浦和レッズ", "kasw": "柏レイソル",
    "tk-v": "東京ヴェルディ", "ka-f": "川崎フロンターレ", "shim": "清水エスパルス",
    "kyot": "京都サンガF.C.", "c-os": "セレッソ大阪", "okay": "岡山",
    "fuku": "アビスパ福岡", "mito": "水戸ホーリーホック", "chib": "ジェフユナイテッド千葉",
    "FCtk": "FC東京", "mcd": "FC町田ゼルビア", "y-fm": "横浜FM",
    "nago": "名古屋グランパス", "g-os": "ガンバ大阪", "kobe": "ヴィッセル神戸",
    "hiro": "サンフレッチェ広島", "ngsk": "Ｖ・ファーレン長崎", "y-fc": "横浜FC",
    "shon": "湘南ベルマーレ", "niig": "アルビレックス新潟", "iwat": "ジュビロ磐田",
    "sapp": "北海道コンサドーレ札幌", "tosu": "サガン鳥栖", "toku": "徳島ヴォルティス",
    "oita": "大分トリニータ", "send": "ベガルタ仙台"
}

# TEAM_NAME_MAP：略称→フル日本語名
TEAM_NAME_MAP = {
    '札幌': '北海道コンサドーレ札幌', '鹿島': '鹿島アントラーズ', '浦和': '浦和レッズ',
    '柏': '柏レイソル', 'FC東京': 'FC東京', '東京Ｖ': '東京ヴェルディ', '町田': 'FC町田ゼルビア',
    '川崎Ｆ': '川崎フロンターレ', '横浜FM': '横浜FM', '横浜FC': '横浜FC',
    '湘南': '湘南ベルマーレ', '新潟': 'アルビレックス新潟', '清水': '清水エスパルス',
    '磐田': 'ジュビロ磐田', '名古屋': '名古屋グランパス', '京都': '京都サンガF.C.',
    'Ｇ大阪': 'ガンバ大阪', 'Ｃ大阪': 'セレッソ大阪', '神戸': 'ヴィッセル神戸',
    '広島': 'サンフレッチェ広島', '徳島': '徳島ヴォルティス', '福岡': 'アビスパ福岡',
    '鳥栖': 'サガン鳥栖', '大分': '大分トリニータ', '仙台': 'ベガルタ仙台'
}

# 日本語名→コードへの逆マッピングを作成
JAPANESE_TO_CODE = {v: k for k, v in TEAMS.items()}

# HomeとAwayの列を日本語名から略語（コード）に変換
# TEAM_NAME_MAPを使って略称をフル日本語名に正規化してからコードに変換
def normalize_and_convert_team(team_name):
    """チーム名を正規化してコードに変換"""
    # まずTEAM_NAME_MAPでフル名に変換を試みる
    normalized = TEAM_NAME_MAP.get(team_name, team_name)
    # JAPANESE_TO_CODEでコードに変換
    code = JAPANESE_TO_CODE.get(normalized, None)
    return code if code else team_name  # コードが見つからない場合は元の名前を返す

# HomeとAwayを変換
ML_dataset['Home'] = ML_dataset['Home'].apply(normalize_and_convert_team)
ML_dataset['Away'] = ML_dataset['Away'].apply(normalize_and_convert_team)

print("✓ チーム名をコードに変換しました")
print(ML_dataset[['Home', 'Away']].head(10))

✓ チーム名をコードに変換しました
   Home  Away
0  ka-f  y-fm
1  uraw  FCtk
2  sapp  y-fc
3  oita  toku
4  hiro  send
5  kasm  shim
6  shon  tosu
7  c-os  kasw
8  kobe  g-os
9  fuku  nago


In [7]:
# ML_datasetをCSVとして保存
ML_dataset.to_csv('/Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv', index=False)
print("✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv")

✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv


# 1 チーム基礎実力

In [8]:
# 順位に関する特徴量を作成

# チーム名マッピングを適用（df_ranksのチーム名をML_datasetのコードに統一）
# df_ranksのチーム名をML_datasetのHome/Away列の形式に合わせる
team_name_to_code = {
    'ＦＣ東京': 'fctk', 'アビスパ福岡': 'fuku', 'ガンバ大阪': 'g-os', 'サガン鳥栖': 'tosu',
    'サンフレッチェ広島': 'hiro', 'セレッソ大阪': 'c-os', 'ベガルタ仙台': 'send',
    'ヴィッセル神戸': 'kobe', '北海道コンサドーレ札幌': 'sapp', '名古屋グランパス': 'nago',
    '大分トリニータ': 'oita', '川崎フロンターレ': 'ka-f', '徳島ヴォルティス': 'toku',
    '柏レイソル': 'kasw', '横浜Ｆ・マリノス': 'y-fm', '浦和レッズ': 'uraw',
    '清水エスパルス': 'shim', '湘南ベルマーレ': 'shon', '鹿島アントラーズ': 'kasm',
    '横浜ＦＣ': 'y-fc', 'アルビレックス新潟': 'niig', '京都サンガF.C.': 'kyot',
    'ジュビロ磐田': 'iwat', '水戸ホーリーホック': 'mito', 'ジェフユナイテッド千葉': 'chib',
    'ＦＣ町田ゼルビア': 'mcd', 'Ｖ・ファーレン長崎': 'ngsk', 'ファジアーノ岡山': 'okay', '東京ヴェルディ': 'tk-v'
}

df_ranks['Team_Code'] = df_ranks['Team'].map(team_name_to_code)

# Current Rank: 試合前の順位（Section - 1の順位）
def get_prev_rank(team_code, season, section):
    """試合前の順位を取得"""
    if section <= 1:
        return None
    prev_section = section - 1
    rank_data = df_ranks[
        (df_ranks['Team_Code'] == team_code) &
        (df_ranks['Year'] == season) &
        (df_ranks['Section'] == prev_section)
    ]
    return rank_data['Rank'].iloc[0] if not rank_data.empty else None

ML_dataset['Home_Current_Rank'] = ML_dataset.apply(
    lambda row: get_prev_rank(row['Home'], row['Season'], row['Section']), axis=1
)
ML_dataset['Away_Current_Rank'] = ML_dataset.apply(
    lambda row: get_prev_rank(row['Away'], row['Season'], row['Section']), axis=1
)

# Rank Diff: Home_Current_Rank - Away_Current_Rank
ML_dataset['Rank_Diff'] = ML_dataset['Home_Current_Rank'] - ML_dataset['Away_Current_Rank']

# Rank Delta 3: 直近3節での順位変動（現在の順位 - 3節前の順位）
def get_rank_delta_3(team_code, season, section):
    """直近3節での順位変動を計算"""
    if section <= 3:
        return None
    current_rank_data = df_ranks[
        (df_ranks['Team_Code'] == team_code) &
        (df_ranks['Year'] == season) &
        (df_ranks['Section'] == section)
    ]
    prev_rank_data = df_ranks[
        (df_ranks['Team_Code'] == team_code) &
        (df_ranks['Year'] == season) &
        (df_ranks['Section'] == section - 3)
    ]
    if not current_rank_data.empty and not prev_rank_data.empty:
        return current_rank_data['Rank'].iloc[0] - prev_rank_data['Rank'].iloc[0]
    return None

ML_dataset['Home_Rank_Delta_3'] = ML_dataset.apply(
    lambda row: get_rank_delta_3(row['Home'], row['Season'], row['Section']), axis=1
)
ML_dataset['Away_Rank_Delta_3'] = ML_dataset.apply(
    lambda row: get_rank_delta_3(row['Away'], row['Season'], row['Section']), axis=1
)

# NaN値を0で埋める
ML_dataset['Home_Current_Rank'] = ML_dataset['Home_Current_Rank'].fillna(0)
ML_dataset['Away_Current_Rank'] = ML_dataset['Away_Current_Rank'].fillna(0)
ML_dataset['Rank_Diff'] = ML_dataset['Rank_Diff'].fillna(0)
ML_dataset['Home_Rank_Delta_3'] = ML_dataset['Home_Rank_Delta_3'].fillna(0)
ML_dataset['Away_Rank_Delta_3'] = ML_dataset['Away_Rank_Delta_3'].fillna(0)

In [9]:
# Elo Ratingの計算関数（高度なアルゴリズム）
def calculate_elo_ratings(df_matches):

    K = 32
    INITIAL_ELO = 1500
    HOME_ADVANTAGE = 100

    elo_ratings = {}
    current_year = None

    df_sorted = df_matches.sort_values(['Season', 'Section']).copy()

    for idx, match in df_sorted.iterrows():

        year = match['Season']  # Season列を使用
        home_team = match['Home']  # Home列を使用
        away_team = match['Away']  # Away列を使用
        home_score = match['Home_Goals']  # Home_Goals列を使用
        away_score = match['Away_Goals']  # Away_Goals列を使用

        # -----------------------------
        # ③ シーズンリセット
        # -----------------------------
        if current_year is None:
            current_year = year

        if year != current_year:
            for team in elo_ratings:
                elo_ratings[team] = 0.75 * elo_ratings[team] + 0.25 * INITIAL_ELO
            current_year = year

        # 初期Elo
        if home_team not in elo_ratings:
            elo_ratings[home_team] = INITIAL_ELO
        if away_team not in elo_ratings:
            elo_ratings[away_team] = INITIAL_ELO

        home_elo = elo_ratings[home_team]
        away_elo = elo_ratings[away_team]

        # -----------------------------
        # ① ホームアドバンテージ
        # -----------------------------
        home_elo_adj = home_elo + HOME_ADVANTAGE

        expected_home = 1 / (1 + 10 ** ((away_elo - home_elo_adj) / 400))
        expected_away = 1 - expected_home

        # -----------------------------
        # 実際の結果
        # -----------------------------
        if home_score > away_score:
            actual_home = 1
            actual_away = 0
        elif home_score == away_score:
            actual_home = 0.5
            actual_away = 0.5
        else:
            actual_home = 0
            actual_away = 1

        # -----------------------------
        # ② 得失点差補正
        # -----------------------------
        goal_diff = abs(home_score - away_score)

        if goal_diff <= 1:
            multiplier = 1
        elif goal_diff == 2:
            multiplier = 1.5
        else:
            multiplier = (11 + goal_diff) / 8

        K_adj = K * multiplier

        # -----------------------------
        # Elo更新
        # -----------------------------
        new_home_elo = home_elo + K_adj * (actual_home - expected_home)
        new_away_elo = away_elo + K_adj * (actual_away - expected_away)

        elo_ratings[home_team] = new_home_elo
        elo_ratings[away_team] = new_away_elo

        # 試合前のレーティングを保存
        df_sorted.loc[idx, 'Home_Elo_Before'] = home_elo
        df_sorted.loc[idx, 'Away_Elo_Before'] = away_elo
        
        df_sorted.loc[idx, 'Home_Elo_Rating'] = new_home_elo
        df_sorted.loc[idx, 'Away_Elo_Rating'] = new_away_elo

    return df_sorted

# Elo Ratingを計算してML_datasetに適用
print("高度なElo Ratingを計算中...")
ML_dataset_with_elo = calculate_elo_ratings(ML_dataset)
ML_dataset = ML_dataset_with_elo

# Elo Ratingの差を計算（Home - Away）
ML_dataset['Elo_Diff'] = ML_dataset['Home_Elo_Before'] - ML_dataset['Away_Elo_Before']

print("✓ 高度なElo Ratingを追加しました")
print(ML_dataset[['Home', 'Away', 'Home_Elo_Before', 'Away_Elo_Before', 'Elo_Diff']].head(10))

高度なElo Ratingを計算中...
✓ 高度なElo Ratingを追加しました
   Home  Away  Home_Elo_Before  Away_Elo_Before  Elo_Diff
0  ka-f  y-fm           1500.0           1500.0       0.0
1  uraw  FCtk           1500.0           1500.0       0.0
2  sapp  y-fc           1500.0           1500.0       0.0
3  oita  toku           1500.0           1500.0       0.0
4  hiro  send           1500.0           1500.0       0.0
5  kasm  shim           1500.0           1500.0       0.0
6  shon  tosu           1500.0           1500.0       0.0
7  c-os  kasw           1500.0           1500.0       0.0
8  kobe  g-os           1500.0           1500.0       0.0
9  fuku  nago           1500.0           1500.0       0.0


In [10]:
# ML_datasetにdf_market_valuesのTotalMarketValue_M_Euroを「Market Value」カラムとして結合する 

# チームコードとチーム名のマッピング（コード→日本語名）

tm_to_code = {
    "Urawa Red Diamonds": "uraw",
    "Vissel Kobe": "kobe",
    "Kawasaki Frontale": "ka-f",
    "Yokohama F. Marinos": "y-fm",
    "FC Tokyo": "FCtk",
    "Hokkaido Consadole Sapporo": "sapp",
    "Yokohama FC": "y-fc",
    "Oita Trinita": "oita",
    "Tokushima Vortis": "toku",
    "Sanfrecce Hiroshima": "hiro",
    "Vegalta Sendai": "send",
    "Kashima Antlers": "kasm",
    "Nagoya Grampus": "nago",
    "Shimizu S-Pulse": "shim",
    "Shonan Bellmare": "shon",
    "Kashiwa Reysol": "kasw",
    "Gamba Osaka": "g-os",
    "Cerezo Osaka": "c-os",
    "Avispa Fukuoka": "fuku",
    "Sagan Tosu": "tosu",
    "Kyoto Sanga": "kyot",
    "Albirex Niigata": "niig",
    "Machida Zelvia": "mcd",
    "Júbilo Iwata": "iwat",
    "Tokyo Verdy": "tk-v",
    "Fagiano Okayama": "okay"
}
df_market_values["Team_Code"] = df_market_values["Team_TM"].map(tm_to_code)

# Homeチームの市場価値を結合
ML_dataset = ML_dataset.merge(
    df_market_values[["Season","Team_Code","TotalMarketValue_M_Euro"]],
    left_on=["Season","Home"],
    right_on=["Season","Team_Code"],
    how="left"
)

ML_dataset = ML_dataset.rename(columns={
    "TotalMarketValue_M_Euro": "Home_Market_Value"
}).drop(columns=["Team_Code"])

# Awayチームの市場価値を結合
ML_dataset = ML_dataset.merge(
    df_market_values[["Season","Team_Code","TotalMarketValue_M_Euro"]],
    left_on=["Season","Away"],
    right_on=["Season","Team_Code"],
    how="left"
)

ML_dataset = ML_dataset.rename(columns={
    "TotalMarketValue_M_Euro": "Away_Market_Value"
}).drop(columns=["Team_Code"])


# ホームチーム、アウェイチームの差分を取る
ML_dataset["Market_Value_Diff"] = (
    ML_dataset["Home_Market_Value"] -
    ML_dataset["Away_Market_Value"]
)

print(ML_dataset[['Home', 'Away', 'Home_Market_Value', 'Away_Market_Value', 'Market_Value_Diff']].head(10))

   Home  Away  Home_Market_Value  Away_Market_Value  Market_Value_Diff
0  ka-f  y-fm              24.15              21.53               2.62
1  uraw  FCtk              25.35              18.15               7.20
2  sapp  y-fc              14.48              12.65               1.83
3  oita  toku              12.20              16.45              -4.25
4  hiro  send              17.85              12.25               5.60
5  kasm  shim              21.43              20.23               1.20
6  shon  tosu              12.05              13.08              -1.03
7  c-os  kasw              21.18              19.10               2.08
8  kobe  g-os              25.03              18.75               6.28
9  fuku  nago              15.90              21.30              -5.40


In [11]:
# マネージャーの就任履歴を作成し、Manager Tenureを計算する関数
def calculate_manager_tenure(df_matches, df_managers, df_manager_changes):
    """
    各試合でのホーム/アウェイチームの現在のマネージャーの就任後の経過日数を計算
    
    Parameters:
    - df_matches: 試合データ（Date, Home, Away, Seasonが必要）
    - df_managers: シーズン開始時のマネージャーデータ
    - df_manager_changes: 監督交代データ
    
    Returns:
    - Manager Tenure列を追加したDataFrame
    """
    
    # 各シーズンの開始日をdf_matchesから取得（Section=1の最小日付）
    season_start_dates = df_matches[df_matches['Section'] == 1].groupby('Season')['Date'].min()
    season_start_dates = {season: pd.Timestamp(date) for season, date in season_start_dates.items()}
    
    # 各チームのマネージャー就任履歴を作成
    manager_histories = {}
    
    # 1. df_managersからシーズン開始時のマネージャーを追加
    for _, row in df_managers.iterrows():
        team_code = team_name_to_code.get(row['Team'])
        if team_code:
            year = row['Year']
            # シーズン開始日をデータから取得
            start_date = season_start_dates.get(year)
            if start_date is None:
                # フォールバック：データがない場合は2月1日を使用
                start_date = pd.Timestamp(f'{year}-02-01')
            
            if team_code not in manager_histories:
                manager_histories[team_code] = []
            
            manager_histories[team_code].append({
                'appointment_date': start_date,
                'manager': row['Manager']
            })
    
    # 2. df_manager_changesから交代情報を追加
    for _, row in df_manager_changes.iterrows():
        team_code = team_name_to_code.get(row['Team'])
        if team_code and pd.notna(row['Appointment_Date']):
            year = row['Year']
            
            # 日付文字列をパース（例: "4月8日" → "2021-04-08"）
            date_str = row['Appointment_Date']
            if '月' in date_str and '日' in date_str:
                month_day = date_str.replace('月', '-').replace('日', '')
                full_date_str = f'{year}-{month_day}'
                try:
                    appointment_date = pd.Timestamp(full_date_str)
                except:
                    continue  # 日付パースエラーの場合はスキップ
            else:
                continue
            
            if team_code not in manager_histories:
                manager_histories[team_code] = []
            
            manager_histories[team_code].append({
                'appointment_date': appointment_date,
                'manager': row['New_Manager'] if pd.notna(row['New_Manager']) else row['Interim_Manager']
            })
    
    # 各チームの履歴を日付順にソート
    for team_code in manager_histories:
        manager_histories[team_code].sort(key=lambda x: x['appointment_date'])
    
    # 3. 各試合に対してManager Tenureを計算
    df_result = df_matches.copy()
    df_result['Home_Manager_Tenure'] = 0
    df_result['Away_Manager_Tenure'] = 0
    
    for idx, match in df_result.iterrows():
        match_date = pd.Timestamp(match['Date'])
        home_team = match['Home']
        away_team = match['Away']
        
        # Homeチームのマネージャー就任日を特定
        if home_team in manager_histories:
            home_appointments = manager_histories[home_team]
            # 試合日以前の最新の就任日を特定
            valid_appointments = [app for app in home_appointments if app['appointment_date'] <= match_date]
            if valid_appointments:
                latest_appointment = max(valid_appointments, key=lambda x: x['appointment_date'])
                tenure_days = (match_date - latest_appointment['appointment_date']).days
                df_result.loc[idx, 'Home_Manager_Tenure'] = tenure_days
        
        # Awayチームのマネージャー就任日を特定
        if away_team in manager_histories:
            away_appointments = manager_histories[away_team]
            # 試合日以前の最新の就任日を特定
            valid_appointments = [app for app in away_appointments if app['appointment_date'] <= match_date]
            if valid_appointments:
                latest_appointment = max(valid_appointments, key=lambda x: x['appointment_date'])
                tenure_days = (match_date - latest_appointment['appointment_date']).days
                df_result.loc[idx, 'Away_Manager_Tenure'] = tenure_days
    
    return df_result

# Manager Tenureを計算してML_datasetに適用
print("マネージャーの就任履歴を作成中...")
ML_dataset_with_tenure = calculate_manager_tenure(ML_dataset, df_managers, df_manager_changes)
ML_dataset = ML_dataset_with_tenure

print("✓ Manager Tenureを追加しました")
print(ML_dataset[['Home', 'Away', 'Date', 'Home_Manager_Tenure', 'Away_Manager_Tenure']].head(10))

マネージャーの就任履歴を作成中...
✓ Manager Tenureを追加しました
   Home  Away        Date  Home_Manager_Tenure  Away_Manager_Tenure
0  ka-f  y-fm  2021-02-26                    0                    0
1  uraw  FCtk  2021-02-27                    1                    0
2  sapp  y-fc  2021-02-27                    1                    0
3  oita  toku  2021-02-27                    1                    1
4  hiro  send  2021-02-27                    1                    1
5  kasm  shim  2021-02-27                    1                    1
6  shon  tosu  2021-02-27                    1                    1
7  c-os  kasw  2021-02-27                    1                    1
8  kobe  g-os  2021-02-27                    1                    1
9  fuku  nago  2021-02-28                    2                    2


In [12]:
# # 各カラムの欠損数（合計）を表示
# print(ML_dataset.isnull().sum())

# 欠損があるカラムのみを表示したい場合
null_counts = ML_dataset.isnull().sum()
print(null_counts[null_counts > 0])

Series([], dtype: int64)


In [13]:
# ML_datasetをML_datasetとしてCSVで保存
ML_dataset.to_csv('/Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv', index=False)
print("✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv")

✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv


# 2 チームの勢い＆直接対決

In [14]:
# 勝ち点（Home_Current_Points, Away_Current_Points）を追加

# チーム名をML_datasetのHome/Awayに合わせて変換
# 注意: このセルを再実行するとき、上から2回目は Team が既にコード化済みのため
# map が全て NaN になる。必ず CSV から読み直す。
# _df_ss = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_season_stats.csv')
df_season_stats['Team'] = df_season_stats['Team'].map(team_name_to_code)

# ML_dataset用に年度・節でマージするための前処理
# ML_dataset: Season, Section, Home, Away, ...（既存）
# それぞれホーム・アウェイごとに "直前" セクションまでの勝ち点を付与（したがって、(節-1)の値を参照）

def get_current_points(df_season_stats, year, team_code, section):
    """
    指定したチーム・年度の直前節までの勝ち点を返す。1節の場合は0。
    """
    if section == 1:
        return 0
    previous = df_season_stats[(df_season_stats['Year'] == year) &
                               (df_season_stats['Section'] == (section - 1)) &
                               (df_season_stats['Team'] == team_code)]
    if previous.empty:
        return 0  # データがなければ0
    return previous.iloc[0]["Points"]

# Home, Away両方について、直前節までの勝ち点を計算して新しいカラムとして追加
home_points = []
away_points = []

for idx, row in ML_dataset.iterrows():
    season = row['Season'] if 'Season' in row.index else row['Year']
    section = row['Section']
    home_code = row['Home']
    away_code = row['Away']
    home_points.append(get_current_points(df_season_stats, season, home_code, section))
    away_points.append(get_current_points(df_season_stats, season, away_code, section))

ML_dataset['Home_Current_Points'] = home_points
ML_dataset['Away_Current_Points'] = away_points
print("✓ 勝ち点（Home_Current_Points, Away_Current_Points）を追加しました")

✓ 勝ち点（Home_Current_Points, Away_Current_Points）を追加しました


In [15]:
# Rolling Points 5: 各チームについて「この試合のキックオフ時点」での直近5試合の獲得勝ち点合計（不足分は0扱い）
# 試合は Date → Season → Section の順で時系列に並べ、各チームの過去の試合結果だけを累積する

def _match_points_from_row(row):
    hg, ag = row["Home_Goals"], row["Away_Goals"]
    if pd.isna(hg) or pd.isna(ag):
        return 0, 0
    if hg > ag:
        return 3, 0
    if hg < ag:
        return 0, 3
    return 1, 1


def add_rolling_points_5(df):
    """ホーム・アウェイそれぞれ Rolling Points 5 を付与（df を上書きして返す）"""
    from collections import defaultdict

    team_hist = defaultdict(list)  # team -> 時系列の各試合で得た勝ち点リスト
    home_roll = {}
    away_roll = {}

    order = df.sort_values(["Date", "Season", "Section"], kind="mergesort")
    for orig_i, row in order.iterrows():
        h, a = row["Home"], row["Away"]
        hp, ap = _match_points_from_row(row)
        h_hist = team_hist[h]
        a_hist = team_hist[a]
        home_roll[orig_i] = sum(h_hist[-5:])
        away_roll[orig_i] = sum(a_hist[-5:])
        h_hist.append(hp)
        a_hist.append(ap)

    df = df.copy()
    df["Home_Rolling_Points_5"] = df.index.map(home_roll)
    df["Away_Rolling_Points_5"] = df.index.map(away_roll)
    return df


ML_dataset = add_rolling_points_5(ML_dataset)
print("✓ Home_Rolling_Points_5 / Away_Rolling_Points_5 を追加しました（直近5試合の勝ち点合計・試合前時点）")



✓ Home_Rolling_Points_5 / Away_Rolling_Points_5 を追加しました（直近5試合の勝ち点合計・試合前時点）


In [16]:
# --- 1. 試合ごとのxGマスタを作成 ---
# 過去データ用：Seasonごとの総xGを節数で割って、全節分に展開する
df_xg['Team'] = df_xg['Team_FL'].map(team_name_to_code)
season_max_sections = ML_dataset.groupby('Season')['Section'].max().to_dict()

match_xg_records = []
for _, row in df_xg.iterrows():
    season = row['Season']
    team = row['Team']
    total_xg = row['Expected_Goals']
    max_sec = season_max_sections.get(season, 34)
    per_match_xg = total_xg / max_sec
    
    for sec in range(1, max_sec + 1):
        match_xg_records.append({
            'Season': season,
            'Section': sec,
            'Team': team,
            'Match_xG': per_match_xg  # ここを実際の毎節のxGに入れ替えれば、自動的にローリングに反映されます
        })

df_match_xg = pd.DataFrame(match_xg_records)

# --- 2. 直近5試合の期待値 (Rolling_xG_5) を計算する関数 ---
def get_rolling_xg(stats_df, year, team_code, section, window=5):
    """
    指定した節の「直前window試合」の平均xGを返す。
    """
    if section <= 1:
        return 0.0
    
    # 当該チームのそのシーズン、直前節までのデータを取得
    history = stats_df[
        (stats_df['Season'] == year) & 
        (stats_df['Section'] < section) & 
        (stats_df['Team'] == team_code)
    ].sort_values('Section')
    
    if history.empty:
        return 0.0
    
    # 直近5試合分を平均（5試合未満ならある分だけで平均）
    return history['Match_xG'].tail(window).mean()

# --- 3. ML_dataset への適用 ---
home_rolling_xg = []
away_rolling_xg = []

for idx, row in ML_dataset.iterrows():
    season = row['Season'] if 'Season' in row.index else row['Year']
    section = row['Section']
    
    h_xg = get_rolling_xg(df_match_xg, season, row['Home'], section)
    a_xg = get_rolling_xg(df_match_xg, season, row['Away'], section)
    
    home_rolling_xg.append(h_xg)
    away_rolling_xg.append(a_xg)

ML_dataset['Home_Rolling_xG'] = home_rolling_xg
ML_dataset['Away_Rolling_xG'] = away_rolling_xg

print("✓ 直近5試合平均xG（Home_Rolling_xG, Away_Rolling_xG）を追加しました")

✓ 直近5試合平均xG（Home_Rolling_xG, Away_Rolling_xG）を追加しました


In [17]:
# # --- 検証用コード ---
# CHECK_TEAM = 'uraw'  # 確認したいチーム名
# CHECK_SEASON = 2021  # 確認したいシーズン

# # 1. ML_datasetから対象チームの試合を抽出
# # 自分がHomeの時は Home_Rolling_xG、Awayの時は Away_Rolling_xG を「自分のRolling値」として取得
# check_df = ML_dataset[
#     (ML_dataset['Season'] == CHECK_SEASON) & 
#     ((ML_dataset['Home'] == CHECK_TEAM) | (ML_dataset['Away'] == CHECK_TEAM))
# ].sort_values('Section').copy()

# # 2. 比較のために、元データの「1試合あたりのxG（定数値）」を紐付ける
# # df_match_xg には全節分の Match_xG が入っているので、それを参照
# check_df = pd.merge(
#     check_df,
#     df_match_xg[df_match_xg['Team'] == CHECK_TEAM][['Season', 'Section', 'Match_xG']],
#     on=['Season', 'Section'],
#     how='left'
# )

# # 3. 表示用に「自分のRolling値」を一つのカラムにまとめる
# check_df['My_Rolling_xG'] = check_df.apply(
#     lambda r: r['Home_Rolling_xG'] if r['Home'] == CHECK_TEAM else r['Away_Rolling_xG'],
#     axis=1
# )

# # 4. 結果の表示
# display_cols = ['Season', 'Section', 'Home', 'Away', 'Match_xG', 'My_Rolling_xG']
# print(f"--- {CHECK_TEAM} の xG ローリング計算確認 ({CHECK_SEASON}年) ---")
# print(check_df[display_cols].to_string(index=False))

# # 5. 計算ロジックの正当性チェック
# print("\n[判定メモ]")
# print(f"・第1節の My_Rolling_xG が 0.0 であること: {'OK' if check_df.iloc[0]['My_Rolling_xG'] == 0 else 'NG'}")
# if len(check_df) > 1:
#     expected = check_df.iloc[0]['Match_xG']
#     actual = check_df.iloc[1]['My_Rolling_xG']
#     print(f"・第2節の My_Rolling_xG が 第1節の Match_xG ({expected:.3f}) と一致すること: {'OK' if abs(actual - expected) < 1e-5 else 'NG'}")

In [18]:
# --- AGI / KAGI 特徴量の追加 ---

# 1. チーム名の変換（修復済みの team_name_to_code を使用）
df_agi_kagi['Team'] = df_agi_kagi['Team_FL'].map(team_name_to_code)

# 2. シーズンごとの最大節数を取得（J1: 34 or 38節など）
season_max_sections = ML_dataset.groupby('Season')['Section'].max().to_dict()

# 3. 1試合あたりの AGI / KAGI を算出
def calc_per_game_metrics(df):
    results = []
    for _, row in df.iterrows():
        season = row['Season']
        max_sec = season_max_sections.get(season, 34) # 取得できない場合はデフォルト34
        
        results.append({
            'Season': season,
            'Team': row['Team'],
            'AGI_per_game': row['AGI'] / max_sec,
            'KAGI_per_game': row['KAGI'] / max_sec
        })
    return pd.DataFrame(results)

df_metrics_features = calc_per_game_metrics(df_agi_kagi)

# 4. ML_dataset への結合（Home側）
ML_dataset = pd.merge(
    ML_dataset,
    df_metrics_features.rename(columns={
        'Team': 'Home', 
        'AGI_per_game': 'Home_AGI', 
        'KAGI_per_game': 'Home_KAGI'
    }),
    on=['Season', 'Home'],
    how='left'
)

# 5. ML_dataset への結合（Away側）
ML_dataset = pd.merge(
    ML_dataset,
    df_metrics_features.rename(columns={
        'Team': 'Away', 
        'AGI_per_game': 'Away_AGI', 
        'KAGI_per_game': 'Away_KAGI'
    }),
    on=['Season', 'Away'],
    how='left'
)

# 6. 欠損値の補完（データがない昇格チームなどは平均的な値で埋める）
# AGI/KAGIの平均的な値（例: 50前後を節数34で割った1.5付近）で埋めるか、0で埋める
ML_dataset[['Home_AGI', 'Home_KAGI', 'Away_AGI', 'Away_KAGI']] = \
    ML_dataset[['Home_AGI', 'Home_KAGI', 'Away_AGI', 'Away_KAGI']].fillna(0)

print("✓ AGI / KAGI 特徴量を追加しました")

✓ AGI / KAGI 特徴量を追加しました


In [19]:
# --- AGI / KAGI 検証用コード ---
CHECK_TEAM = 'uraw'  # 確認したいチーム名 (浦和)
CHECK_SEASON = 2021  # 確認したいシーズン

# 1. 比較用に、元の df_agi_kagi から「1試合あたりの平均値」を算出しておく
# (前のセルで計算した df_metrics_features を利用)
target_metrics = df_metrics_features[
    (df_metrics_features['Season'] == CHECK_SEASON) & 
    (df_metrics_features['Team'] == CHECK_TEAM)
].iloc[0]

expected_agi = target_metrics['AGI_per_game']
expected_kagi = target_metrics['KAGI_per_game']

# 2. ML_dataset から対象チームの試合を抽出
check_df = ML_dataset[
    (ML_dataset['Season'] == CHECK_SEASON) & 
    ((ML_dataset['Home'] == CHECK_TEAM) | (ML_dataset['Away'] == CHECK_TEAM))
].sort_values(['Section', 'Date']).copy()

# 3. 表示用に「自分のAGI/KAGI」を抽出
check_df['My_AGI'] = check_df.apply(
    lambda r: r['Home_AGI'] if r['Home'] == CHECK_TEAM else r['Away_AGI'],
    axis=1
)
check_df['My_KAGI'] = check_df.apply(
    lambda r: r['Home_KAGI'] if r['Home'] == CHECK_TEAM else r['Away_KAGI'],
    axis=1
)

# 4. 結果の表示
display_cols = ['Season', 'Section', 'Home', 'Away', 'My_AGI', 'My_KAGI']
print(f"--- {CHECK_TEAM} の AGI/KAGI 結合確認 ({CHECK_SEASON}年) ---")
print(f"理論上の1試合平均: AGI={expected_agi:.3f}, KAGI={expected_kagi:.3f}")
print("-" * 60)
print(check_df[display_cols].head(10).to_string(index=False))

# 5. 整合性チェック
actual_agi = check_df['My_AGI'].iloc[0]
print("\n[判定結果]")
if abs(actual_agi - expected_agi) < 1e-5:
    print(f"✅ OK: ML_dataset内の値が、シーズン平均 (AGI:{expected_agi:.3f}) と一致しています。")
else:
    print(f"❌ NG: 値が一致しません。マージ処理を確認してください。")

--- uraw の AGI/KAGI 結合確認 (2021年) ---
理論上の1試合平均: AGI=1.192, KAGI=1.332
------------------------------------------------------------
 Season  Section Home Away   My_AGI  My_KAGI
   2021        1 uraw FCtk 1.192105 1.331579
   2021        2 tosu uraw 1.192105 1.331579
   2021        3 uraw y-fc 1.192105 1.331579
   2021        4 y-fm uraw 1.192105 1.331579
   2021        5 uraw sapp 1.192105 1.331579
   2021        6 uraw ka-f 1.192105 1.331579
   2021        7 uraw kasm 1.192105 1.331579
   2021        8 shim uraw 1.192105 1.331579
   2021        9 uraw toku 1.192105 1.331579
   2021       10 c-os uraw 1.192105 1.331579

[判定結果]
✅ OK: ML_dataset内の値が、シーズン平均 (AGI:1.192) と一致しています。


In [20]:
# --- H2H_Score_Avg 特徴量の追加 ---

# 1. 前準備: 日付型への変換とスコアの数値化
ML_dataset['Date'] = pd.to_datetime(ML_dataset['Date'])

def parse_gd(score_str):
    try:
        h, a = map(int, score_str.split('-'))
        return h - a
    except:
        return 0

# 各試合のホームチームから見た得失点差を一時的に保持
ML_dataset['match_gd'] = ML_dataset['Score'].apply(parse_gd)

# 2. H2H平均を計算する関数の定義
def get_h2h_avg_gd(df, current_date, home_team, away_team, years=3):
    start_date = current_date - pd.DateOffset(years=years)
    
    # 過去3年間の、この2チーム間の全対戦を抽出（ホーム・アウェイ問わず）
    # かつ、今回の試合日より前のデータのみ
    mask = (
        (df['Date'] < current_date) & 
        (df['Date'] >= start_date) &
        (
            ((df['Home'] == home_team) & (df['Away'] == away_team)) |
            ((df['Home'] == away_team) & (df['Away'] == home_team))
        )
    )
    
    h2h_matches = df[mask]
    
    if h2h_matches.empty:
        return 0.0
    
    gds = []
    for _, row in h2h_matches.iterrows():
        # 今回のホームチームが過去戦でホームだった場合
        if row['Home'] == home_team:
            gds.append(row['match_gd'])
        # 今回のホームチームが過去戦でアウェイだった場合（得失点差を反転）
        else:
            gds.append(-row['match_gd'])
            
    return sum(gds) / len(gds)

# 3. ML_dataset への適用（計算に時間がかかる場合があるため apply を使用）
print("H2H特徴量を計算中...")
ML_dataset['H2H_Score_Avg'] = ML_dataset.apply(
    lambda row: get_h2h_avg_gd(ML_dataset, row['Date'], row['Home'], row['Away']),
    axis=1
)

# 不要な一時カラムを削除
ML_dataset = ML_dataset.drop(columns=['match_gd'])

print("✓ H2H_Score_Avg（過去3年の直接対決平均得失点差）を追加しました")

H2H特徴量を計算中...
✓ H2H_Score_Avg（過去3年の直接対決平均得失点差）を追加しました


In [21]:
# --- H2H_Score_Avg 検証用コード (3年制約対応版) ---
TEAM_A = 'uraw'
TEAM_B = 'ka-f'
YEARS_WINDOW = 3

# 1. ML_datasetから対象カードを抽出
h2h_check = ML_dataset[
    ((ML_dataset['Home'] == TEAM_A) & (ML_dataset['Away'] == TEAM_B)) |
    ((ML_dataset['Home'] == TEAM_B) & (ML_dataset['Away'] == TEAM_A))
].sort_values('Date').copy()

# 2. 過去3年の内訳と平均を計算する関数（検証用）
def verify_h2h_logic(df, current_row):
    current_date = current_row['Date']
    home_team = current_row['Home']
    away_team = current_row['Away']
    start_date = current_date - pd.DateOffset(years=YEARS_WINDOW)
    
    # 過去3年・今回の試合より前の対戦を抽出
    past_matches = df[
        (df['Date'] < current_date) & 
        (df['Date'] >= start_date)
    ]
    
    if past_matches.empty:
        return 0.0, []

    gds = []
    for _, r in past_matches.iterrows():
        # スコアから得失点差をパース (Home - Away)
        h_score, a_score = map(int, r['Score'].split('-'))
        match_gd = h_score - a_score
        
        # 今回のホームチーム視点に変換
        val = match_gd if r['Home'] == home_team else -match_gd
        gds.append(val)
        
    avg = sum(gds) / len(gds)
    return avg, gds

# 3. 最新の試合で詳細検証
print(f"--- {TEAM_A} vs {TEAM_B} の H2H 計算検証 (直近{YEARS_WINDOW}年) ---")
last_match = h2h_check.iloc[-1]
manual_avg, target_gds = verify_h2h_logic(h2h_check, last_match)

# 全体の推移表示
display_cols = ['Date', 'Home', 'Away', 'Score', 'H2H_Score_Avg']
print(h2h_check[display_cols].to_string(index=False))

print(f"\n[最新試合の詳細チェック: {last_match['Date'].date()}]")
print(f"今回のHome: {last_match['Home']} / 今回のAway: {last_match['Away']}")
print(f"計算対象期間: {(last_match['Date'] - pd.DateOffset(years=YEARS_WINDOW)).date()} ～ {last_match['Date'].date()}")
print(f"対象となった過去の得失点差(Home視点): {target_gds}")
print(f"手動計算平均: {manual_avg:.3f}")
print(f"カラムの値  : {last_match['H2H_Score_Avg']:.3f}")

if abs(manual_avg - last_match['H2H_Score_Avg']) < 1e-5:
    print("✅ OK: 3年間の期間制限を含め、計算が一致しました。")
else:
    print("❌ NG: まだ一致しません。")

--- uraw vs ka-f の H2H 計算検証 (直近3年) ---
      Date Home Away Score  H2H_Score_Avg
2021-03-21 uraw ka-f   0-5            0.0
2021-11-03 ka-f uraw   1-1            5.0
2022-03-02 ka-f uraw   2-1            2.5
2022-07-30 uraw ka-f   3-1           -2.0
2023-04-23 ka-f uraw   1-1            1.0
2023-06-24 uraw ka-f   1-1           -0.8
2024-05-03 ka-f uraw   3-1           -0.2
2024-11-22 uraw ka-f   1-1           -0.2
2025-05-21 ka-f uraw   2-2            0.0
2025-12-06 uraw ka-f   4-0           -0.4

[最新試合の詳細チェック: 2025-12-06]
今回のHome: uraw / 今回のAway: ka-f
計算対象期間: 2022-12-06 ～ 2025-12-06
対象となった過去の得失点差(Home視点): [0, 0, -2, 0, 0]
手動計算平均: -0.400
カラムの値  : -0.400
✅ OK: 3年間の期間制限を含め、計算が一致しました。


In [22]:
import unicodedata
import pandas as pd

# --- Stadium Fill Rate 特徴量の追加 (完全網羅・数値クレンジング版) ---

# 1. 前準備：テキストの正規化関数（全角半角・空白・カッコを統一）
def normalize_text(text):
    if not isinstance(text, str): return ""
    # NFKC正規化で全角英数を半角に、カッコ等のゆらぎを統一
    text = unicodedata.normalize('NFKC', text)
    # すべての空白（全角・半角・改行）を除去
    return "".join(text.split())

# 2. マスターデータ(stadium_df)の徹底的なクリーニング
# キャパシティの「15,000」のようなカンマを除去して数値化
stadium_df['Capacity'] = (
    stadium_df['Capacity'].astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)
stadium_df['Capacity'] = pd.to_numeric(stadium_df['Capacity'], errors='coerce')
stadium_df['Clean_Name'] = stadium_df['Stadium_Name'].apply(normalize_text)

# 3. 略称から正式名称（に含まれるキーワード）へのマッピング辞書
# ユーザー様から報告のあった15件＋主要スタジアムをすべて網羅
search_keywords = {
    'レモンＳ': 'レモンガス',
    'ベススタ': 'ベスト電器',
    'アイスタ': 'IAI',
    '鳴門大塚': 'ポカリスエット',
    'ユアスタ': 'ユアテック',
    '駅スタ': '駅前不動産',
    '三協Ｆ柏': '三協フロンテア',
    '神戸ユ': 'ユニバー',
    'サンガＳ': 'サンガ',
    'デンカＳ': 'デンカ',
    'Ｅピース': 'ピースウイング',
    'Ｇスタ': 'GION',
    'プレド': 'プレミスト',
    'ＪＦＥス': 'フクダ電子',  # JFEスチール蘇我の跡地＝フクアリ
    'メルスタ': 'メルカリ',    # カシマスタジアムのメルカリ提携
    '昭和電ド': 'クラサス',
    'Ｅスタ': 'ピースウイング',
    'ヤンマー': 'ヨドコウ',
    'ノエスタ': 'ノエビア',
    '味スタ': '味の素',
    'パナスタ': 'パナソニック',
    'カシマ': 'カシマ',
    '札幌ド': 'プレミスト',
    'Ｕ等々力': 'とどろき',
    '等々力': 'とどろき',
    '埼玉': '埼玉スタジアム',
    'ニッパツ': 'ニッパツ',
    '豊田ス': '豊田',
    'ヨドコウ': 'ヨドコウ',
    'ヤマハ': 'ヤマハ',
    '長良川': '長良川',
    '浦和駒場': '駒場',
    '札幌厚別': '厚別',
    'エコパ': 'エコパ',
    '国立': '国立'
}

# 4. マッチング実行
stadium_to_capacity = {}
for ml_name in ML_dataset['Stadium'].unique():
    # キーワードがあればそれを使用。なければ元の名前を正規化して使う。
    search_word = search_keywords.get(ml_name, ml_name)
    target_word = normalize_text(search_word)
    
    # 部分一致検索（マスターの名称にキーワードが含まれているか）
    match_row = stadium_df[stadium_df['Clean_Name'].str.contains(target_word, na=False)]
    
    if not match_row.empty:
        stadium_to_capacity[ml_name] = match_row.iloc[0]['Capacity']

# 5. カラムの反映と計算
ML_dataset['Capacity'] = ML_dataset['Stadium'].map(stadium_to_capacity)
ML_dataset['Attendance'] = pd.to_numeric(ML_dataset['Attendance'], errors='coerce')

# Fill Rateの算出（CapacityがNaNでない場合のみ）
ML_dataset['Stadium_Fill_Rate'] = ML_dataset['Attendance'] / ML_dataset['Capacity']
ML_dataset['Stadium_Fill_Rate'] = ML_dataset['Stadium_Fill_Rate'].fillna(0)

In [23]:
# Match_Result (勝敗:ホーム勝ち=1、引き分け=0、アウェイ勝ち=-1)と Goal_Diff (得点差)を作成
# Goal_Diff: ホームチーム視点の得点差 (Home_Goals - Away_Goals)
ML_dataset['Goal_Diff'] = ML_dataset['Home_Goals'] - ML_dataset['Away_Goals']

# Match_Result: ホームチームの勝敗 (勝ち=1、引き分け=0、負け=-1)
def get_match_result(goal_diff):
    if pd.isna(goal_diff):
        return 0
    if goal_diff > 0:
        return 1  # ホーム勝ち
    elif goal_diff == 0:
        return 0  # 引き分け
    else:
        return -1  # ホーム負け (アウェイ勝ち)

ML_dataset['Match_Result'] = ML_dataset['Goal_Diff'].apply(get_match_result)


print("✓ Match_Result（ホーム勝ち=1、引き分け=0、アウェイ勝ち=-1）を追加しました")
print("✓ Goal_Diff（ホーム視点の得点差）を追加しました")
print(ML_dataset[['Score', 'Goal_Diff', 'Match_Result']].head(15))

✓ Match_Result（ホーム勝ち=1、引き分け=0、アウェイ勝ち=-1）を追加しました
✓ Goal_Diff（ホーム視点の得点差）を追加しました
   Score  Goal_Diff  Match_Result
0    2-0          2             1
1    1-1          0             0
2    5-1          4             1
3    1-1          0             0
4    1-1          0             0
5    1-3         -2            -1
6    0-1         -1            -1
7    2-0          2             1
8    1-0          1             1
9    1-2         -1            -1
10   2-2          0             0
11   3-2          1             1
12   1-1          0             0
13   1-5         -4            -1
14   1-2         -1            -1


In [24]:
# ML_datasetをML_datasetとしてCSVで保存
ML_dataset.to_csv('/Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv', index=False)
print("✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv")

✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv


# 3 個人の稼働状況＆期待値

こちらは、個人の得点に関するデータなので、03_Bayes_Tableでの得点者の予測に使う。

# 4 チームスタッツ

In [25]:
import pandas as pd

# 1. 各シーズンの最大節数（試合数）の辞書
season_max_sections = ML_dataset.groupby('Season')['Section'].max().to_dict()
if 2020 not in season_max_sections:
    season_max_sections[2020] = 34 

# --- df_team_stats の1試合平均化（前処理） ---

already_avg_cols = [
    'Average Possession', 'Avg Distance Covered', 'Avg Sprints',
    'Distance Covered (In Possession)', 'Sprints (In Possession)',
    'Distance Covered (Out of Possession)', 'Sprints (Out of Possession)',
    'Avg Actual Playing Time' 
]

id_cols = ['year', 'team_name']
stats_cols = [c for c in df_team_stats.columns if c not in id_cols]

# 1試合あたりの平均値を保持するDataFrameを作成
df_team_per_game = df_team_stats.copy()
df_team_per_game['max_sections_in_year'] = df_team_per_game['year'].map(season_max_sections).fillna(34)

for col in stats_cols:
    df_team_per_game[col] = pd.to_numeric(df_team_per_game[col], errors='coerce')
    if col not in already_avg_cols:
        df_team_per_game[col] = df_team_per_game[col] / df_team_per_game['max_sections_in_year']

# チーム名コード変換
df_team_per_game['team_code'] = df_team_per_game['team_name'].map(team_name_to_code)

# --- 昨シーズン用データの準備 ---
df_prev_season = df_team_per_game.copy()
df_prev_season['year'] = df_prev_season['year'] + 1 # 2020年の値を2021年の試合に当てるため
# カラム名を Prev_... に書き換えて重複を防ぐ
prev_rename_dict = {col: f'Prev_{col}' for col in stats_cols}
df_prev_season = df_prev_season.rename(columns=prev_rename_dict)

# --- ML_dataset へのマージと累計計算 ---

ML_dataset['Season'] = ML_dataset['Season'].astype(int)

for side in ['Home', 'Away']:
    # A. 今シーズンの1試合平均データをマージ
    ML_dataset = pd.merge(
        ML_dataset, 
        df_team_per_game.drop(columns=['team_name', 'max_sections_in_year']), 
        left_on=['Season', side], 
        right_on=['year', 'team_code'], 
        how='left'
    )
    
    # B. 昨シーズンの1試合平均データをマージ
    ML_dataset = pd.merge(
        ML_dataset,
        df_prev_season.drop(columns=['team_name', 'max_sections_in_year']),
        left_on=['Season', side],
        right_on=['year', 'team_code'],
        how='left',
        suffixes=('', '_prev_tmp') # 万が一の重複回避
    )
    
    # 特徴量計算
    for col in stats_cols:
        # 今シーズンの特徴量（累計または平均）
        new_curr_col = f'{side}_{col}'
        if col in already_avg_cols:
            ML_dataset[new_curr_col] = ML_dataset[col]
        else:
            # 累計系は (Section - 1) を掛けて、試合直前の累計値を出す
            ML_dataset[new_curr_col] = ML_dataset[col] * (ML_dataset['Section'] - 1)
        
        # 昨シーズンの特徴量（1試合平均）
        new_prev_col = f'{side}_Prev_{col}'
        ML_dataset[new_prev_col] = ML_dataset[f'Prev_{col}']
    
    # 使用した中間カラムを削除
    drop_cols = ['year', 'team_code', 'year_prev_tmp', 'team_code_prev_tmp'] + stats_cols + list(prev_rename_dict.values())
    ML_dataset = ML_dataset.drop(columns=[c for c in drop_cols if c in ML_dataset.columns], errors='ignore')

# 最終的な欠損値埋め（昇格チームなどで昨季データがない場合は0）
ML_dataset = ML_dataset.fillna(0)

display(ML_dataset.head())

,Season,Section,Date,Home,Score,Away,Stadium,Attendance,Home_Goals,Away_Goals,...,Away_Avg Sprints,Away_Prev_Avg Sprints,Away_Distance Covered (In Possession),Away_Prev_Distance Covered (In Possession),Away_Sprints (In Possession),Away_Prev_Sprints (In Possession),Away_Distance Covered (Out of Possession),Away_Prev_Distance Covered (Out of Possession),Away_Sprints (Out of Possession),Away_Prev_Sprints (Out of Possession)
0,2021,1,2021-02-26,ka-f,2-0,y-fm,等々力,4868,2,0,...,212.0,184.0,52.0,51.0,106.0,89.0,44.0,47.0,102.0,93.0
1,2021,1,2021-02-27,uraw,1-1,FCtk,埼玉,4943,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021,1,2021-02-27,sapp,5-1,y-fc,札幌ド,11897,5,1,...,161.0,154.0,38.0,45.0,70.0,72.0,49.0,49.0,88.0,81.0
3,2021,1,2021-02-27,oita,1-1,toku,昭和電ド,7012,1,1,...,159.0,0.0,47.0,0.0,72.0,0.0,44.0,0.0,83.0,0.0
4,2021,1,2021-02-27,hiro,1-1,send,Ｅスタ,8820,1,1,...,165.0,159.0,38.0,38.0,75.0,69.0,51.0,54.0,87.0,89.0


In [26]:
# --- 設定 ---
sample_team_ja = '鹿島アントラーズ'  
sample_team_code = team_name_to_code[sample_team_ja] 
target_col = 'Total Shots'
check_year = 2022
prev_year = 2021

print(f"--- 診断開始: {sample_team_ja} ({sample_team_code}) ---")

df_target = df_team_stats[(df_team_stats['year'] == prev_year) & (df_team_stats['team_name'] == sample_team_ja)]

if df_target.empty:
    print(f"❌ df_team_stats 内に {prev_year}年の {sample_team_ja} が見つかりません。")
else:
    raw_val_prev = df_target[target_col].values[0]
    max_sec_prev = season_max_sections.get(prev_year, 0)
    
    if max_sec_prev == 0:
        print(f"❌ season_max_sections に {prev_year}年 のデータがありません。")
    else:
        # 昨季の1試合平均（期待値）
        expected_prev_avg = raw_val_prev / max_sec_prev
        print(f"📌 [期待値] {prev_year}年の1試合平均: {expected_prev_avg:.4f}\n")

        ml_target = ML_dataset[(ML_dataset['Season'] == check_year) & 
                               ((ML_dataset['Home'] == sample_team_code) | (ML_dataset['Away'] == sample_team_code))]

        if ml_target.empty:
            print(f"❌ ML_dataset 内に {check_year}年の {sample_team_code} の試合が見つかりません。")
        else:
            # 変化を見るために、第1節と第10節の2試合をチェックしてみる
            for test_idx in [0, 9]: # 0=第1節あたり, 9=第10節あたり
                if test_idx >= len(ml_target):
                    continue
                    
                row = ml_target.iloc[test_idx]
                prefix = 'Home_' if row['Home'] == sample_team_code else 'Away_'
                section = row['Section']
                
                # ① 今季の累計値（ Section - 1 を掛けたもの）
                actual_curr_val = row[f'{prefix}{target_col}']
                # ② 昨季の平均値（固定値）
                actual_prev_val = row[f'{prefix}Prev_{target_col}']
                
                print(f"--- 【第{section}節】 ({prefix[:-1]}側) ---")
                print(f"  今季ここまでの累計 ({prefix}{target_col}): {actual_curr_val:.4f}")
                print(f"  昨季の平均実績     ({prefix}Prev_{target_col}): {actual_prev_val:.4f}")
                
                if abs(actual_prev_val - expected_prev_avg) < 1e-7:
                    print("  ✅ 昨季データ照合: 完璧に一致しています！")
                elif actual_prev_val == 0:
                    print("  ❌ 昨季データが0です。マージ失敗の可能性あり。")
                else:
                    print(f"  ❌ 昨季データが不一致。(差分: {actual_prev_val - expected_prev_avg})")
                print("-" * 30)

--- 診断開始: 鹿島アントラーズ (kasm) ---
📌 [期待値] 2021年の1試合平均: 15.2368

--- 【第1節】 (Away側) ---
  今季ここまでの累計 (Away_Total Shots): 0.0000
  昨季の平均実績     (Away_Prev_Total Shots): 15.2368
  ✅ 昨季データ照合: 完璧に一致しています！
------------------------------
--- 【第10節】 (Away側) ---
  今季ここまでの累計 (Away_Total Shots): 116.4706
  昨季の平均実績     (Away_Prev_Total Shots): 15.2368
  ✅ 昨季データ照合: 完璧に一致しています！
------------------------------


In [27]:
# ML_datasetをML_datasetとしてCSVで保存
ML_dataset.to_csv('/Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv', index=False)
print("✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv")

✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv


# 5 コンディション・外部要因（Conditioning & Context）

In [28]:
# --- フォーメーション関連特徴量の追加 ---
# 1. ミラーゲーム判定 (完全一致)
ML_dataset['is_Mirror_Game'] = (ML_dataset['Home_Formation'] == ML_dataset['Away_Formation']).astype(int)

# 2. フォーメーションを解析してDF, MF, FWの人数を抽出する関数
def parse_formation_zones(fmt_str):
    if pd.isna(fmt_str) or not isinstance(fmt_str, str):
        return pd.Series([0, 0, 0])
    
    # '-' で分割して数値リスト化 (例: '4-1-2-3' -> [4, 1, 2, 3])
    parts = [int(x) for x in fmt_str.split('-') if x.isdigit()]
    
    if len(parts) < 3:
        return pd.Series([0, 0, 0])
    
    df = parts[0]       # 最初の数字は常にDF
    fw = parts[-1]      # 最後の数字は常にFW
    mf = sum(parts[1:-1]) # 真ん中の数字はすべてMFとして合算
    
    return pd.Series([df, mf, fw])

# 3. Home / Away それぞれのエリア別人数を算出
ML_dataset[['Home_DF_count', 'Home_MF_count', 'Home_FW_count']] = ML_dataset['Home_Formation'].apply(parse_formation_zones)
ML_dataset[['Away_DF_count', 'Away_MF_count', 'Away_FW_count']] = ML_dataset['Away_Formation'].apply(parse_formation_zones)

# 4. 【重要】噛み合わせ（Formation Match）特徴量の作成
# 中盤の数的優位性 (正の値ならHomeが中盤で人数が多い)
ML_dataset['Home_Midfield_Advantage'] = ML_dataset['Home_MF_count'] - ML_dataset['Away_MF_count']

# 最終ラインの数的余裕度 (HomeのDF人数 - AwayのFW人数)
# 3バックに対して3トップ(差が0)だとピンチになりやすい、などの傾向を捉える
ML_dataset['Defense_Margin_Home'] = ML_dataset['Home_DF_count'] - ML_dataset['Away_FW_count']
ML_dataset['Defense_Margin_Away'] = ML_dataset['Away_DF_count'] - ML_dataset['Home_FW_count']

# 5. (オプション) バックラインシステムの衝突カテゴリ (例: "4_vs_3")
ML_dataset['Backline_Matchup'] = ML_dataset['Home_DF_count'].astype(str) + "_vs_" + ML_dataset['Away_DF_count'].astype(str)

display(ML_dataset[['Home_Formation', 'Away_Formation', 'is_Mirror_Game', 'Home_Midfield_Advantage', 'Backline_Matchup']].head(10))

,Home_Formation,Away_Formation,is_Mirror_Game,Home_Midfield_Advantage,Backline_Matchup
0,4-1-2-3,4-2-3-1,0,-2,4_vs_4
1,4-2-3-1,4-2-3-1,1,0,4_vs_4
2,3-4-2-1,3-4-2-1,1,0,3_vs_3
3,3-4-2-1,4-2-3-1,0,1,3_vs_4
4,3-4-2-1,4-4-2,0,2,3_vs_4
5,4-4-2,4-4-2,1,0,4_vs_4
6,3-3-2-2,3-3-2-2,1,0,3_vs_3
7,4-4-2,3-4-2-1,0,-2,4_vs_3
8,4-4-2,4-4-2,1,0,4_vs_4
9,4-4-2,4-2-3-1,0,-1,4_vs_4


In [29]:
# --- ゴールパターン・スタイル指標の特徴量追加 ---

#  前処理：共通の変換関数
def prepare_static_stats(df, val_cols, is_index=False):
    df = df[df['Category'] == 'J1'].copy()
    df['full_team_name'] = df['Team_Short'].map(TEAM_NAME_MAP)
    df['team_code'] = df['full_team_name'].map(team_name_to_code)
    
    # 1試合平均化（Index系でない場合のみ）
    if not is_index:
        df['max_sections'] = df['Season'].map(season_max_sections).fillna(34)
        for col in val_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce') / df['max_sections']
    
    # 【重要】Seasonを+1して「昨季のデータ」として扱えるようにする
    df_prev = df[['Season', 'team_code'] + val_cols].copy()
    df_prev['Season'] = df_prev['Season'] + 1
    
    # カラム名に Prev_ を付与
    rename_dict = {col: f'Prev_{col}' for col in val_cols}
    return df_prev.rename(columns=rename_dict)

# 2. 各データの準備
goal_cols = [c for c in df_goal_patterns.columns if 'Goal' in c]
style_cols = ['Set-piece Index', 'Left Attack Index', 'Center Attack Index', 'Right Attack Index', 
              'Short Counter', 'Long Counter', 'Opponent Area Possession', 'My Area Possession']

df_goals_final = prepare_static_stats(df_goal_patterns, goal_cols, is_index=False)
df_styles_final = prepare_static_stats(df_team_styles, style_cols, is_index=True)

# 3. ML_dataset へのマージ（昨季データのみ）
for side in ['Home', 'Away']:
    # ゴールパターンのマージ
    ML_dataset = pd.merge(
        ML_dataset, df_goals_final,
        left_on=['Season', side], right_on=['Season', 'team_code'],
        how='left'
    )
    # スタイル指標のマージ
    ML_dataset = pd.merge(
        ML_dataset, df_styles_final,
        left_on=['Season', side], right_on=['Season', 'team_code'],
        how='left', suffixes=('', '_style_tmp')
    )
    
    # Home_/Away_ を付与してリネーム
    rename_map = {}
    for col in list(df_goals_final.columns) + list(df_styles_final.columns):
        if col not in ['Season', 'team_code']:
            rename_map[col] = f'{side}_{col}'
            
    ML_dataset = ML_dataset.rename(columns=rename_map)
    
    # 中間カラムの削除
    ML_dataset = ML_dataset.drop(columns=['team_code', 'team_code_style_tmp'], errors='ignore')

# 欠損値（昇格組など）を0埋め
ML_dataset = ML_dataset.fillna(0)
print("✓ ゴールパターン・スタイル指標の特徴量を追加しました")

✓ ゴールパターン・スタイル指標の特徴量を追加しました


# 6 シーズン状況

In [30]:
df_season_stats

,Year,Section,Team,Rank,Points,Season_Progress,Urgency_Score
0,2021,1,sapp,1,3,0.026316,0.000693
1,2021,1,shim,2,3,0.026316,0.000693
2,2021,1,ka-f,3,3,0.026316,0.000693
3,2021,1,c-os,3,3,0.026316,0.000693
4,2021,1,nago,5,3,0.026316,0.000693
...,...,...,...,...,...,...,...
3499,2025,38,nago,16,43,1.000000,0.111111
3500,2025,38,tk-v,17,43,1.000000,0.111111
3501,2025,38,y-fc,18,35,1.000000,1.000000
3502,2025,38,shon,19,32,1.000000,1.000000


In [31]:
# マージに必要なカラムだけを抽出 (Year, Section, Team, 追加したい2つの指標)
# ML_datasetのSeasonと合わせるため、YearをSeasonにリネームしておくとスムーズです
df_progress = df_season_stats[['Year', 'Section', 'Team', 'Season_Progress', 'Urgency_Score']].copy()
df_progress = df_progress.rename(columns={'Year': 'Season'})

# 2. ML_dataset へのマージ
for side in ['Home', 'Away']:
    # マージ実行
    ML_dataset = pd.merge(
        ML_dataset,
        df_progress,
        left_on=['Season', 'Section', side],
        right_on=['Season', 'Section', 'Team'],
        how='left'
    )
    
    # カラム名に接頭辞（Home_ / Away_）を付与
    ML_dataset = ML_dataset.rename(columns={
        'Season_Progress': f'{side}_Season_Progress',
        'Urgency_Score': f'{side}_Urgency_Score'
    })
    
    # マージ用のテンポラリカラムを削除
    if 'Team' in ML_dataset.columns:
        ML_dataset = ML_dataset.drop(columns=['Team'])

# 欠損値（もしあれば）を0埋め
ML_dataset[['Home_Season_Progress', 'Away_Season_Progress', 'Home_Urgency_Score', 'Away_Urgency_Score']] = \
    ML_dataset[['Home_Season_Progress', 'Away_Season_Progress', 'Home_Urgency_Score', 'Away_Urgency_Score']].fillna(0)

print("✅ Season Progress および Urgency Score のマージが完了しました。")

# 3. 該当カラムの確認
check_cols = ['Season', 'Section', 'Home', 'Away', 
              'Home_Season_Progress', 'Away_Season_Progress', 
              'Home_Urgency_Score', 'Away_Urgency_Score']
display(ML_dataset[check_cols].head())

✅ Season Progress および Urgency Score のマージが完了しました。


,Season,Section,Home,Away,Home_Season_Progress,Away_Season_Progress,Home_Urgency_Score,Away_Urgency_Score
0,2021,1,ka-f,y-fm,0.026316,0.026316,0.000693,0.000693
1,2021,1,uraw,FCtk,0.026316,0.000000,0.000346,0.000000
2,2021,1,sapp,y-fc,0.026316,0.026316,0.000693,0.000693
3,2021,1,oita,toku,0.026316,0.026316,0.000346,0.000346
4,2021,1,hiro,send,0.026316,0.026316,0.000346,0.000346


In [32]:
# ML_datasetをCSVとして保存
ML_dataset.to_csv('/Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv', index=False)
print("✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv")

✓ ML_dataset を保存しました: /Users/akihirookuyama/Soccer_Score_App/Data/ML_dataset.csv
